# EfficientNet-B0 acoustic baseline

Select the repository `.venv` kernel after `uv sync --extra notebooks --extra audio --extra deep-learning`. Run `data_prep.ipynb` first; paths and W&B credentials come from `.env` or the environment.

Set parameters below, then run the training cell. `SMOKE_RUN=True` uses a small training subset for one epoch and keeps the full validation fold. Set it to `False` for a full run in a separate directory.

The model trains a mono EfficientNet-B0 encoder and dropout/linear head on five-second Log-Mel inputs. `pretrained=True` downloads ImageNet weights on first use. Restoring a checkpoint needs no download. See `src/birdclef/models/audio_cnn/README.md` for the data protocol and architecture.


In [ ]:
import json
import os

import torch

from birdclef.config import RuntimeConfig, TrainConfig, get_config
from birdclef.models import AudioExperimentConfig, AudioModelConfig

RUN_NAME = "audio-cnn"
SMOKE_RUN = True
WANDB_OFFLINE = False  # True to log locally and sync later with wandb sync.
CPU_THREADS = 4

torch.set_num_threads(CPU_THREADS)

config = get_config(
    AudioExperimentConfig,
    fold=0,
    smoke=SMOKE_RUN,
    model=AudioModelConfig(pretrained=True),
    tracking={"offline": WANDB_OFFLINE},
    runtime=RuntimeConfig(accelerator="auto", devices=1),
    loader={
        "batch_size": 4 if SMOKE_RUN else 64,
        "val_batch_size": 8,
        "predict_batch_size": 12,
        "num_workers": 0,
        "seed": 42,
    },
    train=TrainConfig(
        max_epochs=1 if SMOKE_RUN else 5,
        checkpoint_metric="val/auc",
        checkpoint_mode="max",
        early_stopping_patience=3,
        log_every_n_steps=1,
    ),
)
RUN_DIR = config.data.output_dir / config.run_name(RUN_NAME)
print(f"Run directory: {RUN_DIR}")


## Train

`%run` executes the shared entry point in the notebook kernel, so logs and progress appear directly in the cell. Use one device here; interrupt the cell to stop training.

The entry point loads prepared data, trains or resumes the checkpoint, evaluates the best model and logs W&B metrics and artifacts. Re-running continues the same run; increasing `max_epochs` keeps its name. Changed data or training settings require a separate run.

The saved config excludes API keys. Use `WANDB_API_KEY` in the environment or `BCLEF_TRACKING__API_KEY` in `.env`.


In [ ]:
if config.tracking.api_key is not None:
    os.environ["BCLEF_TRACKING__API_KEY"] = config.tracking.api_key.get_secret_value()

config_path = config.data.output_dir / "configs" / f"{config.run_name(RUN_NAME)}.json"
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(json.dumps(config.safe_dump(), indent=2), encoding="utf-8")


In [ ]:
%run -m birdclef.models.audio_cnn.fit -- --config "{config_path}" --name "{RUN_NAME}"


## Validation results

The reports use the best checkpoint and the entire saved validation fold. Classes without positives have undefined per-class AUC and are skipped in the macro score. Smoke scores only check the pipeline. This is one fold, not pooled OOF.


In [ ]:
import pandas as pd
from IPython.display import display

display(pd.read_json(RUN_DIR / "validation-metrics.json", typ="series"))
display(
    pd.read_csv(RUN_DIR / "validation-class-metrics.csv")
    .sort_values("auc", ascending=False)
    .head(15)
)


## Optional: CPU inference and a sanity-check CSV

Restore the saved model on CPU and predict all 12 windows of one validation soundscape. Timing includes checkpoint loading, decoding, Log-Mel and inference. The CSV checks submission formatting; it is not a competition submission or a model-quality result.


In [ ]:
import time

from birdclef.competition import ROW_ID, soundscape_windows
from birdclef.models import AudioCNN
from birdclef.submission import build_submission, write_submission

cpu_config = config.model_copy(deep=True)
cpu_config.runtime = RuntimeConfig(accelerator="cpu", devices=1, precision="32-true")
restored = AudioCNN.from_run(RUN_DIR, config=cpu_config)
classes = restored.classes
all_windows = soundscape_windows(config.data.competition_dir / "train_soundscapes")
validation_ids = pd.read_csv(RUN_DIR / "validation-predictions.csv", usecols=[ROW_ID])[ROW_ID]
filename = all_windows.loc[all_windows[ROW_ID].isin(validation_ids), "filename"].iloc[0]
minute = all_windows.loc[all_windows.filename.eq(filename)].reset_index(drop=True)
assert len(minute) == 12
started = time.perf_counter()
probabilities = restored.predict(minute)
elapsed = time.perf_counter() - started

template = pd.DataFrame(0.0, index=range(len(minute)), columns=classes)
template.insert(0, ROW_ID, minute[ROW_ID].to_numpy())
submission = build_submission(probabilities, template)
path = write_submission(submission, RUN_DIR / "minute-sanity-submission.csv", template)
benchmark = {
    "seconds": elapsed,
    "audio_seconds": 60,
    "windows": len(minute),
    "cpu_threads": CPU_THREADS,
    "torch_version": torch.__version__,
    "includes": "checkpoint load, Trainer setup, decode, Log-Mel, CNN",
}
(RUN_DIR / "cpu-benchmark.json").write_text(json.dumps(benchmark, indent=2), encoding="utf-8")
display(pd.Series(benchmark))
display(submission.head())
print(f"Sanity-check CSV: {path}")
